<a href="https://colab.research.google.com/github/ColinaAndres/Heart-Disease-Prediction/blob/main/notebooks/Red_Neuronal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports y autenticacion

In [ ]:
import numpy as np
import random
import os
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import initializers
from keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
!pip install feature-engine


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.6/378.6 kB 5.9 MB/s eta 0:00:00


In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
id = '1QHrv5jOfyW6znzSvY8U4eGgLsAWa_Ln0'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('heart_train.csv')

In [ ]:
id_test = '1ToTRlYQqwxhNn48T5JvIoEFxCxDmZRpR'
downloaded_test = drive.CreateFile({'id': id_test})
downloaded_test.GetContentFile('heart_test.csv')

# Carga Dataset

In [ ]:
heart_data = pd.read_csv('heart_train.csv',encoding='latin1')
test_data = pd.read_csv('heart_test.csv',encoding='latin1')

# Red Neuronal

## Random States

Segun la documentacion de keras, es necesario controlar todas las variables aleatorias globales para la que el modelo sea reproducible en el entorno, ademas hay que desactivar el shuffle en el training

In [ ]:
seed = 15
keras.utils.set_random_seed(seed)

Hace que los calculos sean lo mas deterministicos **posibles**

In [ ]:
tf.config.experimental.enable_op_determinism()

## Division train validation

In [ ]:
x = heart_data.drop('HeartDisease', axis=1)
y = heart_data['HeartDisease']

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_validation, y_train, y_validation = train_test_split(x, y, test_size=0.2, random_state=seed)

## Preprocesamiento

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from feature_engine.encoding import MeanEncoder

In [ ]:
oh_encoding_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
mean_encoding_cols = ['RestingECG', 'ChestPainType']
norm_encoding_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'FastingBS']

### OneHotEncoding

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
encoder.fit(x_train[oh_encoding_cols])

x_train_oh = encoder.transform(x_train[oh_encoding_cols])
x_validation_oh = encoder.transform(x_validation[oh_encoding_cols])
test_oh = encoder.transform(test_data[oh_encoding_cols])

### MeanEncoding

In [ ]:
mean_enc = MeanEncoder(variables=mean_encoding_cols)
mean_enc.fit(x_train, y_train)

x_train_mean = mean_enc.transform(x_train)
x_validation_mean = mean_enc.transform(x_validation)
test_mean = mean_enc.transform(test_data)

x_train_mean = x_train_mean[mean_encoding_cols].values
x_validation_mean = x_validation_mean[mean_encoding_cols].values
test_mean = test_mean[mean_encoding_cols].values

# Concateno antes de escalar asi tengo en consideracion estas tambien.
x_train_to_scale = np.concatenate([x_train[norm_encoding_cols].values, x_train_mean], axis=1)
x_validation_to_scale = np.concatenate([x_validation[norm_encoding_cols].values, x_validation_mean], axis=1)
test_to_scale = np.concatenate([test_data[norm_encoding_cols].values, test_mean], axis=1)


### Normalizacion

In [ ]:
scaler = StandardScaler()
scaler.fit(x_train_to_scale)

x_train_norm = scaler.transform(x_train_to_scale)
x_validation_norm = scaler.transform(x_validation_to_scale)
test_norm = scaler.transform(test_to_scale)

### Concatenacion

In [ ]:
x_train_final = np.concatenate([x_train_oh, x_train_norm], axis=1)
x_validation_final = np.concatenate([x_validation_oh, x_validation_norm], axis=1)
test_final = np.concatenate([test_oh, test_norm], axis=1)

## Armado del modelo

In [ ]:
from keras.initializers import GlorotUniform
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

Usando un inicializador con una seed fija, se puede reproducir el train del modelo, ademas del random state hecho antes para todas las seeds del sistema, particularmente se usa GlorotUniform porque es buena con relu y sigmoid

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(x_train_final.shape[1],)),
    layers.Dense(64, activation='relu', kernel_initializer=GlorotUniform(seed=seed)),
    layers.Dense(32, activation='relu', kernel_initializer=GlorotUniform(seed=seed)),
    Dropout(0.2),
    layers.Dense(16, activation='relu', kernel_initializer=GlorotUniform(seed=seed)),
    layers.Dense(8, activation='relu', kernel_initializer=GlorotUniform(seed=seed)),
    layers.Dense(1, activation='sigmoid', kernel_initializer=GlorotUniform(seed=seed))
])

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'], jit_compile=False)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    x_train_final, y_train,
    validation_data=(x_validation_final, y_validation),
    epochs=200, #despues de 10 epochs sin mejorar early_stop deberia salir.
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5224 - loss: 0.7037 - val_accuracy: 0.7483 - val_loss: 0.5915
Epoch 2/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7747 - loss: 0.5712 - val_accuracy: 0.8095 - val_loss: 0.4658
Epoch 3/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8221 - loss: 0.4523 - val_accuracy: 0.8095 - val_loss: 0.3984
Epoch 4/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8248 - loss: 0.3896 - val_accuracy: 0.8231 - val_loss: 0.3776
Epoch 5/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8317 - loss: 0.3705 - val_accuracy: 0.8299 - val_loss: 0.3653
Epoch 6/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8352 - loss: 0.3587 - val_accuracy: 0.8367 - val_loss: 0.3550
Epoch 7/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8455 - loss: 0.3627 - val_accuracy: 0.8503 - val_loss: 0.3488
Epoch 8/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8588 - loss: 0.3512 - val_accuracy: 0.

## Evaluacion de validacion

In [ ]:
model.evaluate(x_validation_final, y_validation)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8554 - loss: 0.3114 


[0.3270578384399414, 0.8435373902320862]

# Score para la Competencia

In [ ]:
prediccion = model.predict(test_final)
prediccion = (prediccion > 0.5).astype(int)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


In [ ]:
prediccion = prediccion.ravel()
prediccion

array([1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1,
       0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1,
       1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0,
       1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 1, 1, 1])

In [ ]:
submission_red_neuronal = pd.DataFrame({'id': range(len(prediccion)), 'HeartDisease': prediccion})

In [ ]:
submission_red_neuronal.to_csv('submission_red_neuronal.csv', index=False)